# MRMS \u00d7 HydroFabric \u2192 dHBV \u2014 Notebook 3 of 4: Event Manifest, MRMS Download & Fractional Crosswalk

**This notebook builds everything needed before rainfall can be extracted:**
1. **Per-event manifest (gage-basin method).** For each of the HUC8's flood events,
   find its recording gage (nearest gage downstream by network hops) and that
   gage's upstream contributing catchments \u2014 the confirmed, correct catchment
   association rule (footprint-radius buffering was tried and rejected: the
   `footprint_radius_km` field is report-scatter, not rain extent, and collapses
   to ~1 catchment per storm).
2. **Unique MRMS timestamp list.** Every event gets a 5-day window (midpoint
   \u00b1 2.5 days) at 2-min resolution; overlapping events share timestamps, so we
   download the UNION once.
3. **MRMS download**, using `mrms_bbox_downloader.py`'s `build_store()` \u2014
   file-level parallel download, in-memory GRIB decode, bbox subset, resumable
   day-shard NetCDF output. This is the confirmed production downloader (not the
   hand-rolled per-file loop from earlier drafts).
4. **Fractional (area-weighted) MRMS\u2192catchment crosswalk**, restricted to just
   the gage-basin catchments from step 1 \u2014 `fraction_inside` = area(MRMS cell
   \u2229 catchment) / area(MRMS cell), the input to area-weighted rainfall.

**Not in this notebook:** turning downloaded rainfall into per-catchment
2-min \u2192 15-min depth series and exporting NetCDF forcing \u2014 that's
**Notebook 4** (spatial area-weighting, then temporal aggregation, kept
separate since it's a materially different kind of work: reading GRIB/NetCDF
data instead of building spatial reference tables).

**Requires** `mrms_bbox_downloader.py` in the working directory.

## 0. Load Notebook 1 & 2 outputs

In [1]:
import os
import sys
import warnings
from pathlib import Path

proj_data = Path(sys.prefix) / "share" / "proj"
if proj_data.exists():
    os.environ["PROJ_DATA"] = str(proj_data)
    os.environ["PROJ_LIB"] = str(proj_data)

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import pyproj

pyproj.network.set_network_enabled(False)
warnings.filterwarnings("ignore")

# --- Notebook 1 outputs ---
NB1_DIR = Path("hydrofabric_outputs")
assert NB1_DIR.exists(), f"\u274c {NB1_DIR} not found \u2014 run Notebook 1 first."

catchments_master = gpd.read_parquet(NB1_DIR / "catchments_master.parquet")
network           = pd.read_parquet(NB1_DIR / "network.parquet")
flowpaths         = gpd.read_parquet(NB1_DIR / "flowpaths.parquet")
nexus             = gpd.read_parquet(NB1_DIR / "nexus.parquet")
target_crs = catchments_master.crs

# --- Notebook 2 outputs ---
NB2_DIR = Path("huc8_selection_outputs")
assert NB2_DIR.exists(), f"\u274c {NB2_DIR} not found \u2014 run Notebook 2 first."

selected_huc8   = gpd.read_parquet(NB2_DIR / "selected_huc8.parquet")
huc8_catchments = gpd.read_parquet(NB2_DIR / "huc8_catchments.parquet")
events          = pd.read_csv(NB2_DIR / "events_flagged.csv")
valid_storms    = pd.read_csv(NB2_DIR / "valid_storms.csv")
gages_indexed   = pd.read_csv(NB2_DIR / "gages_indexed.csv")

with open(NB2_DIR / "huc8_code.txt") as f:
    huc8_code = f.read().strip()

# --- CONUS crosswalk from Notebook 2 (center-based, one MRMS cell -> one catchment) ---
CACHE = Path("mrms_crosswalk_cache")
COMBINED = CACHE / "mrms_hf_crosswalk_conus.parquet"
assert COMBINED.exists(), f"\u274c {COMBINED} not found \u2014 run Notebook 2's Step 2 first."
crosswalk = pd.read_parquet(COMBINED)

# --- Rebuild the routing graph G (cheap -- Notebook 2 built the same thing, but
#     we don't try to serialize a NetworkX graph between notebooks) ---
edges = network.dropna(subset=["id", "toid"]).copy()
edges = edges[(edges["id"].astype(str).str.strip() != "") &
              (edges["toid"].astype(str).str.strip() != "")]
G = nx.from_pandas_edgelist(edges, source="id", target="toid", create_using=nx.DiGraph())

print(f"HUC8: {huc8_code} | events: {len(events):,} | valid_storms: {len(valid_storms):,} | "
      f"gages_indexed: {len(gages_indexed):,}")
print(f"crosswalk (CONUS, center-based): {len(crosswalk):,} rows")
print(f"G: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

HUC8: 03020201 | events: 5,076 | valid_storms: 99 | gages_indexed: 5,897
crosswalk (CONUS, center-based): 8,647,251 rows
G: 1,233,091 nodes, 1,233,090 edges


# Section 1 \u2014 Per-event manifest (gage-basin method)

For each storm: find all gages **downstream** of it (`nx.descendants`), rank by
fewest network hops, keep the nearest \u2014 that's the **recording gage**. That
gage's **upstream contributing catchments** (`nx.ancestors`) are the catchments
associated with the event (cached per gage so repeated gages aren't recomputed).
Each event also gets a 5-day window centered on its midpoint.

**Method B (storm-footprint buffering by `footprint_radius_km`) is deliberately
not used** \u2014 that field is report-scatter (median \u2248 0.3 km, smaller than one
MRMS cell), so buffering collapses each event to ~1 catchment. Gage-basin is
the confirmed-correct method.

In [2]:
# ============================================================
# Per-event manifest: 5-day window (midpoint \u00b12.5d) + associated
# catchments (= upstream contributing catchments of the event's
# recording gage). Totals the (event, catchment) timeseries count.
# ============================================================

WINDOW_DAYS = 5.0
HALF = pd.to_timedelta(WINDOW_DAYS / 2, unit="D")   # 2.5 days each side

# flowpath id (wb-) -> divide_id, for turning upstream flowpaths into catchments
fp_to_divide = (
    network[["id", "divide_id"]].dropna()
    .assign(id=lambda d: d["id"].astype(str))
    .drop_duplicates(subset="id").set_index("id")["divide_id"]
)

# -----------------------------
# 1. Event begin/end -> midpoint -> 5-day window
# -----------------------------
ev = events.drop_duplicates(subset="storm_index").copy()
ev["begin_time"] = pd.to_datetime(ev["begin"], errors="coerce", utc=True)
ev["end_time"]   = pd.to_datetime(ev["end"],   errors="coerce", utc=True)
ev["midpoint"]   = ev["begin_time"] + (ev["end_time"] - ev["begin_time"]) / 2
ev["win_start"]  = ev["midpoint"] - HALF
ev["win_end"]    = ev["midpoint"] + HALF
mid_lookup = ev.set_index("storm_index")[["begin_time", "end_time",
                                          "midpoint", "win_start", "win_end"]]

# -----------------------------
# 2. Recording gage per storm = nearest gage downstream (fewest hops)
#    + cache each unique gage's upstream catchment set
# -----------------------------
gage_upstream_cats = {}   # gage_flowpath_id -> set(divide_id)

def upstream_cats_of(gfp):
    if gfp in gage_upstream_cats:
        return gage_upstream_cats[gfp]
    up = nx.ancestors(G, gfp); up.add(gfp)
    up_wb = {str(n) for n in up if str(n).startswith("wb-")}
    cats = set(fp_to_divide[fp_to_divide.index.isin(up_wb)].astype(str))
    gage_upstream_cats[gfp] = cats
    return cats

rows = []
no_gage = 0
for _, s in valid_storms.iterrows():
    sidx = s["storm_index"]
    sfp = s["storm_flowpath_id"]
    if sfp not in G:
        no_gage += 1
        continue

    down = nx.descendants(G, sfp); down.add(sfp)
    cand = gages_indexed[gages_indexed["gage_flowpath_id"].isin(down)].copy()
    if not len(cand):
        no_gage += 1
        continue
    cand["hops"] = cand["gage_flowpath_id"].apply(
        lambda fp: nx.shortest_path_length(G, sfp, fp))
    grec = cand.sort_values("hops").iloc[0]

    cats = upstream_cats_of(grec["gage_flowpath_id"])
    win = mid_lookup.loc[sidx] if sidx in mid_lookup.index else None

    rows.append({
        "storm_index": sidx,
        "recording_gage_STAID": grec["STAID"],
        "gage_flowpath_id": grec["gage_flowpath_id"],
        "n_catchments": len(cats),
        "midpoint":  None if win is None else win["midpoint"],
        "win_start": None if win is None else win["win_start"],
        "win_end":   None if win is None else win["win_end"],
        "_divide_ids": sorted(cats),
    })

manifest = pd.DataFrame(rows)

# -----------------------------
# 3. Per-event summary + grand total
# -----------------------------
print(f"Events with a recording gage : {len(manifest):,} "
      f"(no gage downstream: {no_gage})")
print(f"Unique recording gages       : {manifest['recording_gage_STAID'].nunique():,}")
print(f"Total (event, catchment) timeseries segments : "
      f"{manifest['n_catchments'].sum():,}")
print(f"  = sum over events of (catchments in its gage basin)\n")

print("Per-event catchment counts (first 10):")
print(manifest[["storm_index", "recording_gage_STAID",
                "n_catchments", "win_start", "win_end"]]
      .head(10).to_string(index=False))

# -----------------------------
# 4. Long-form (event, catchment, window) table -- the actual list
#    of timeseries to build (drops the bulky _divide_ids column)
# -----------------------------
event_catchment_windows = (
    manifest.explode("_divide_ids")
    .rename(columns={"_divide_ids": "divide_id"})
    .dropna(subset=["divide_id"])
    [["storm_index", "recording_gage_STAID", "divide_id",
      "midpoint", "win_start", "win_end"]]
    .reset_index(drop=True)
)
print(f"\nLong-form rows (event \u00d7 catchment): {len(event_catchment_windows):,}")

manifest_out = manifest.drop(columns="_divide_ids")
print("\nSaved -> manifest (per event), event_catchment_windows (long form)")

Events with a recording gage : 99 (no gage downstream: 0)
Unique recording gages       : 18
Total (event, catchment) timeseries segments : 2,193
  = sum over events of (catchments in its gage basin)

Per-event catchment counts (first 10):
 storm_index  recording_gage_STAID  n_catchments                 win_start                   win_end
        1355               2088500            75 2021-10-07 01:37:30+00:00 2021-10-12 01:37:30+00:00
        1356             208726005            29 2021-10-06 22:18:00+00:00 2021-10-11 22:18:00+00:00
        1385             208758850            16 2021-10-06 22:59:00+00:00 2021-10-11 22:59:00+00:00
        1494               2087275            32 2021-10-06 23:27:30+00:00 2021-10-11 23:27:30+00:00
        1688               2087359             7 2021-10-06 21:30:00+00:00 2021-10-11 21:30:00+00:00
        2875               2087324            42 2021-09-07 00:07:30+00:00 2021-09-12 00:07:30+00:00
        2876               2087324            42 2021-

## 1.1 Export a gage/event CSV (gage, start, end, tag)

One row per (event, recording gage), using each event's 5-day window and the
event's `cluster_id` as the tag. A gage with multiple events gets multiple rows.

In [3]:
gage_event_table = (
    manifest_out
    .merge(
        events[["storm_index", "cluster_id"]].drop_duplicates(subset="storm_index"),
        on="storm_index",
        how="left",
    )
    .assign(
        gage=lambda d: "gage-" + d["recording_gage_STAID"].astype(str),
        start=lambda d: pd.to_datetime(d["win_start"]).dt.strftime("%Y-%m-%d"),
        end=lambda d: pd.to_datetime(d["win_end"]).dt.strftime("%Y-%m-%d"),
        tag=lambda d: d["cluster_id"],
    )
    [["gage", "start", "end", "tag"]]
    .sort_values(["gage", "start"])
    .reset_index(drop=True)
)

GAGE_EVENT_CSV = Path("gage_event_table.csv")
gage_event_table.to_csv(GAGE_EVENT_CSV, index=False)

print(f"rows: {len(gage_event_table):,} | unique gages: {gage_event_table['gage'].nunique():,}")
print(f"saved: {GAGE_EVENT_CSV.resolve()}")
gage_event_table.head()

rows: 99 | unique gages: 18
saved: /home/jovyan/Group_Project/gage_event_table.csv


,gage,start,end,tag
0,gage-2085000,2024-01-07,2024-01-12,519
1,gage-2085000,2025-05-27,2025-06-01,789
2,gage-2085000,2025-07-07,2025-07-12,2761
3,gage-2085000,2025-07-17,2025-07-22,2691
4,gage-2085070,2022-05-21,2022-05-26,655


In [14]:
from pathlib import Path
import pandas as pd

# Merge manifest with event cluster_id
tmp = (
    manifest_out
    .merge(
        events[["storm_index", "cluster_id"]].drop_duplicates(subset="storm_index"),
        on="storm_index",
        how="left",
    )
    .assign(
        gage=lambda d: "gage-" + d["recording_gage_STAID"].astype(str),
        win_start_dt=lambda d: pd.to_datetime(d["win_start"]),
        win_end_dt=lambda d: pd.to_datetime(d["win_end"]),
    )
)

# For each gage, find earliest start, latest end, and cluster_id of earliest storm
gage_event_table = (
    tmp.sort_values(["gage", "win_start_dt"])
    .groupby("gage", as_index=False)
    .agg(
        earliest_start=("win_start_dt", "min"),
        latest_end=("win_end_dt", "max"),
        tag=("cluster_id", "first"),
    )
    .assign(
        start=lambda d: (d["earliest_start"] - pd.DateOffset(months=1)).dt.strftime("%Y-%m-%d"),
        end=lambda d: (d["latest_end"] + pd.DateOffset(months=1)).dt.strftime("%Y-%m-%d"),
    )
    [["gage", "start", "end", "tag"]]
    .sort_values("gage")
    .reset_index(drop=True)
)

GAGE_EVENT_CSV = Path("gage_event_table.csv")
gage_event_table.to_csv(GAGE_EVENT_CSV, index=False)

print(f"rows: {len(gage_event_table):,} | unique gages: {gage_event_table['gage'].nunique():,}")
print(f"saved: {GAGE_EVENT_CSV.resolve()}")
gage_event_table.head()

rows: 18 | unique gages: 18
saved: /home/jovyan/Group_Project/gage_event_table.csv


,gage,start,end,tag
0,gage-2085000,2023-12-07,2025-08-22,519
1,gage-2085070,2022-04-21,2025-08-09,655
2,gage-208521324,2025-04-27,2025-08-18,1103
3,gage-2085220,2025-04-27,2025-07-01,1105
4,gage-2085500,2024-08-22,2025-08-09,2508


# Section 2 \u2014 Unique MRMS timestamp list

Each event window is expanded to 2-min steps, then we take the UNION across
all events \u2014 overlapping windows share timestamps, so a shared file only gets
downloaded once.

In [4]:
MRMS_FREQ = "2min"

mw = manifest.dropna(subset=["win_start", "win_end"]).copy()
mw["win_start"] = pd.to_datetime(mw["win_start"], utc=True).dt.floor(MRMS_FREQ)
mw["win_end"]   = pd.to_datetime(mw["win_end"],   utc=True).dt.ceil(MRMS_FREQ)

# expand every event window to 2-min steps -> long (event, time) table
rows = []
for _, r in mw.iterrows():
    times = pd.date_range(r["win_start"], r["win_end"], freq=MRMS_FREQ)
    rows.append(pd.DataFrame({"storm_index": r["storm_index"], "mrms_time": times}))

event_time_manifest = pd.concat(rows, ignore_index=True)

# the deduplicated set of timestamps to actually download
unique_mrms_times = (
    event_time_manifest["mrms_time"]
    .drop_duplicates().sort_values().reset_index(drop=True)
)

print(f"Events expanded            : {mw['storm_index'].nunique():,}")
print(f"(event, time) rows         : {len(event_time_manifest):,}")
print(f"UNIQUE timestamps to fetch : {len(unique_mrms_times):,}")
print(f"Dedup saving               : "
      f"{1 - len(unique_mrms_times)/len(event_time_manifest):.1%} fewer files")
print(f"Time span                  : {unique_mrms_times.min()} \u2192 {unique_mrms_times.max()}")

Events expanded            : 99
(event, time) rows         : 356,565
UNIQUE timestamps to fetch : 121,456
Dedup saving               : 65.9% fewer files
Time span                  : 2021-09-06 23:54:00+00:00 → 2025-08-17 11:50:00+00:00


# Section 3 \u2014 Download MRMS PrecipRate via `mrms_bbox_downloader.py`

`build_store()` parallelizes over individual 2-min files (not days), decodes
each GRIB2 **in memory** (the full-CONUS grid is never written to disk), and
subsets to the HUC8 bounding box before writing one NetCDF **day shard** per
UTC day under `<out_dir>/shards/`. It's resumable: rerun to fill gaps, and
confirmed-missing timestamps (404 in both archives) are remembered and skipped
next time \u2014 so this cell is safe to interrupt and re-run.

In [5]:
import sys
!{sys.executable} -m pip install -q numpy pandas xarray cfgrib eccodes s3fs fsspec geopandas tqdm


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [6]:
from pathlib import Path
from mrms_bbox_downloader import bbox_from_huc8, build_store, consolidate

# bbox padded 0.1 deg around the selected HUC8 (lon/lat, EPSG:4326)
bbox = bbox_from_huc8(selected_huc8, margin_deg=0.1)
print("bbox (lonmin, latmin, lonmax, latmax):", bbox)

# Absolute path -- pinning this matters. A relative path like "storm_precip_MRMS"
# resolves against whatever the notebook's current working directory happens to
# be, which can differ between kernel restarts / how you launched Jupyter.
MRMS_OUT_DIR = Path("storm_precip_MRMS").resolve()
SHARD_DIR = MRMS_OUT_DIR / "shards"
print("MRMS_OUT_DIR:", MRMS_OUT_DIR)

# Safety net: build_store() compares these timestamps against tz-naive
# np.datetime64 values already on disk. If unique_mrms_times is still
# tz-aware (e.g. this cell is re-run without re-running the timestamp cell
# above), the "already stored" check silently fails and everything
# re-downloads. Strip tz here too, defensively, regardless of upstream state.
if getattr(unique_mrms_times.dt, "tz", None) is not None:
    print("⚠️ unique_mrms_times was tz-aware -- stripping tz to match the "
          "tz-naive shard store.")
    unique_mrms_times = unique_mrms_times.dt.tz_localize(None)

# Pre-flight check: don't just count existing shard files -- actually try to
# open each one with xarray, the same way build_store()'s internal
# _read_existing() does. If a file is unreadable (truncated write, older/
# incompatible encoding, etc.), _read_existing() silently catches the
# exception and treats that day as if it had never been downloaded, which is
# what causes a "files exist on disk but everything re-downloads anyway"
# situation. Surfacing that here, before the download starts, turns a
# confusing multi-hour re-download into an immediate, readable error list.
import xarray as xr

existing_shards = sorted(SHARD_DIR.glob("pr_*.nc")) if SHARD_DIR.exists() else []
existing_bytes = sum(f.stat().st_size for f in existing_shards)
print(f"Existing day shard files: {len(existing_shards):,} "
      f"({existing_bytes / 1e9:.2f} GB) in {SHARD_DIR}")

readable, broken = 0, []
n_times_found = 0
for f in existing_shards:
    try:
        with xr.open_dataset(f) as ds:
            n_times_found += ds.sizes.get("time", 0)
        readable += 1
    except Exception as e:
        broken.append((f.name, repr(e)))

print(f"Readable: {readable:,} / {len(existing_shards):,} "
      f"({n_times_found:,} timestamps total)")

if not existing_shards:
    print("  -> none found at this path. If you expected existing files, "
          "double check MRMS_OUT_DIR matches the folder used previously.")
elif broken:
    print(f"  ⚠️ {len(broken):,} file(s) failed to open -- this is almost "
          "certainly why build_store() thinks nothing is stored and "
          "re-downloads. First few failures:")
    for name, err in broken[:5]:
        print(f"    - {name}: {err}")
elif readable and n_times_found == 0:
    print("  ⚠️ Files open fine but contain 0 timestamps -- likely empty/"
          "placeholder shards from an interrupted run.")
else:
    print("  ✅ All existing shards are readable and contain data -- "
          "build_store() below should resume, not re-download.")

summary = build_store(
    unique_mrms_times,
    bbox,
    str(MRMS_OUT_DIR),
    max_workers=12,
    mask_negative=True,   # PrecipRate -3 ("no radar coverage") -> NaN
    use_aws=True,         # AWS noaa-mrms-pds first, Iowa State HTTPS fallback
    verbose=True,
)
print(summary)
print(f"\nDay shards: {SHARD_DIR.resolve()}")

bbox (lonmin, latmin, lonmax, latmax): (-79.32866436499995, 35.0738186750001, -77.87203142200002, 36.49927568000008)
MRMS_OUT_DIR: /home/jovyan/Group_Project/storm_precip_MRMS
⚠️ unique_mrms_times was tz-aware -- stripping tz to match the tz-naive shard store.
Existing day shard files: 200 (0.38 GB) in /home/jovyan/Group_Project/storm_precip_MRMS/shards
Readable: 200 / 200 (122,641 timestamps total)
  ✅ All existing shards are readable and contain data -- build_store() below should resume, not re-download.
Skipping 278 timestamps already known missing.
121178 timestamps across 196 UTC days | 118973 already stored, 2205 to fetch -> /home/jovyan/Group_Project/storm_precip_MRMS/shards


files:   0%|          | 0/2205 [00:00<?, ?file/s]

Done: {'days': 196, 'to_fetch': 2205, 'stored_new': 2199, 'confirmed_missing': 6, 'transient_errors': 0, 'shard_dir': '/home/jovyan/Group_Project/storm_precip_MRMS/shards', 'manifest': '/home/jovyan/Group_Project/storm_precip_MRMS/build_manifest.csv'}
{'days': 196, 'to_fetch': 2205, 'stored_new': 2199, 'confirmed_missing': 6, 'transient_errors': 0, 'shard_dir': '/home/jovyan/Group_Project/storm_precip_MRMS/shards', 'manifest': '/home/jovyan/Group_Project/storm_precip_MRMS/build_manifest.csv'}

Day shards: /home/jovyan/Group_Project/storm_precip_MRMS/shards


### 3.1 Optional: consolidate into one NetCDF/Zarr store

Not required \u2014 Notebook 4 can read the per-day shards directly and only
needs to open the days each event's window touches. Consolidating everything
into one file is convenient for browsing but duplicates the data on disk.

In [7]:
# consolidate(str(SHARD_DIR), MRMS_OUT_DIR + "/precip_rate.nc", fmt="netcdf")

# Section 4 \u2014 Fractional (area-weighted) MRMS\u2192catchment crosswalk

Restricted to just the gage-basin catchments from Section 1
(`event_catchment_windows['divide_id']`) \u2014 this is the same true-intersection
polygon math as the full-CONUS version, just scoped down so it runs in minutes
instead of hours. `fraction_inside` = area(MRMS cell \u2229 catchment) / area(MRMS
cell); weights are then normalized to sum to 1 per catchment.

In [11]:
# ============================================================
# 4.1 Narrow catchments_master + the center-based crosswalk down to the
#     gage-basin catchments (method-agnostic vs. the footprint-buffer
#     version in the original notebook -- just a different divide_id set).
# ============================================================
selected_divide_ids = event_catchment_windows["divide_id"].dropna().unique()
print(f"Gage-basin catchments (event-relevant): {len(selected_divide_ids):,}")

selected_catchments = catchments_master[
    catchments_master["divide_id"].isin(selected_divide_ids)
].copy()
print(f"selected_catchments rows: {len(selected_catchments):,}")
print(f"selected VPUs: {selected_catchments['vpuid'].nunique():,}")

selected_crosswalk = crosswalk[
    crosswalk["divide_id"].isin(selected_divide_ids)
].copy()
print(f"selected_crosswalk rows: {len(selected_crosswalk):,}")
print(f"unique MRMS cells (center-based): {selected_crosswalk['cell_id'].nunique():,}")

STEP3_CACHE = CACHE / "step3_event_relevant"
STEP3_CACHE.mkdir(exist_ok=True)
selected_catchments.to_parquet(STEP3_CACHE / "selected_catchments.parquet", index=False)

Gage-basin catchments (event-relevant): 293
selected_catchments rows: 293
selected VPUs: 1
selected_crosswalk rows: 2,242
unique MRMS cells (center-based): 2,242


In [12]:
# ============================================================
# 4.2 Build fractional crosswalk for selected (gage-basin) catchments only
#
# Inputs:  selected_catchments, selected_crosswalk, CACHE
# Outputs: selected_fractional_crosswalk (+ .parquet)
# ============================================================

import time
from shapely.geometry import box

# ------------------------------------------------------------
# MRMS grid settings
# ------------------------------------------------------------
MRMS_LON_MIN_EDGE = -130.0
MRMS_LAT_MAX_EDGE = 55.0
MRMS_RES = 0.01
MRMS_NLON = 7000
MRMS_NLAT = 3500
HALF = MRMS_RES / 2.0
AREA_CRS = 5070

# Neighbor padding around center-based MRMS cells
# 1 is usually enough; 2 is safer for edge cases.
PAD_CELLS = 2

FRAC_SELECTED_CACHE = STEP3_CACHE / "selected_fractional_crosswalk_cache"
FRAC_SELECTED_CACHE.mkdir(parents=True, exist_ok=True)
SELECTED_FRAC_FILE = STEP3_CACHE / "selected_fractional_crosswalk.parquet"


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def lon_from_col(col):
    return MRMS_LON_MIN_EDGE + HALF + col * MRMS_RES

def lat_from_row(row):
    return MRMS_LAT_MAX_EDGE - HALF - row * MRMS_RES

def cell_id_from_row_col(row, col):
    return row.astype(np.int64) * MRMS_NLON + col.astype(np.int64)

def build_mrms_cells_from_rows_cols(rows, cols):
    rows = np.asarray(rows, dtype=np.int32)
    cols = np.asarray(cols, dtype=np.int32)

    lon = lon_from_col(cols)
    lat = lat_from_row(rows)

    gdf = gpd.GeoDataFrame(
        {
            "cell_id": cell_id_from_row_col(rows, cols),
            "row": rows,
            "col": cols,
            "lon": lon,
            "lat": lat,
            "geometry": [
                box(x - HALF, y - HALF, x + HALF, y + HALF)
                for x, y in zip(lon, lat)
            ]
        },
        geometry="geometry",
        crs="EPSG:4326"
    ).to_crs(AREA_CRS)

    gdf["cell_area_m2"] = gdf.geometry.area

    return gdf

def rows_cols_for_bbox(west, south, east, north, pad=2):
    i0 = int(np.floor((west - MRMS_LON_MIN_EDGE - HALF) / MRMS_RES)) - pad
    i1 = int(np.ceil((east - MRMS_LON_MIN_EDGE - HALF) / MRMS_RES)) + pad

    j0 = int(np.floor((MRMS_LAT_MAX_EDGE - HALF - north) / MRMS_RES)) - pad
    j1 = int(np.ceil((MRMS_LAT_MAX_EDGE - HALF - south) / MRMS_RES)) + pad

    i0 = max(0, i0)
    i1 = min(MRMS_NLON - 1, i1)

    j0 = max(0, j0)
    j1 = min(MRMS_NLAT - 1, j1)

    if i1 < i0 or j1 < j0:
        return pd.DataFrame(columns=["row", "col"])

    cols = np.arange(i0, i1 + 1)
    rows = np.arange(j0, j1 + 1)

    COL, ROW = np.meshgrid(cols, rows)

    return pd.DataFrame({
        "row": ROW.ravel().astype(np.int32),
        "col": COL.ravel().astype(np.int32)
    })

def add_neighbor_cells(df, pad=2):
    base = df[["row", "col"]].drop_duplicates().copy()

    pieces = []

    for dr in range(-pad, pad + 1):
        for dc in range(-pad, pad + 1):
            tmp = base.copy()
            tmp["row"] = tmp["row"] + dr
            tmp["col"] = tmp["col"] + dc
            pieces.append(tmp)

    out = pd.concat(pieces, ignore_index=True)

    out = out[
        out["row"].between(0, MRMS_NLAT - 1) &
        out["col"].between(0, MRMS_NLON - 1)
    ]

    return out.drop_duplicates().reset_index(drop=True)

def intersection_area_chunked(candidates, chunk_size=150_000):
    areas = []

    for start in range(0, len(candidates), chunk_size):
        end = min(start + chunk_size, len(candidates))
        chunk = candidates.iloc[start:end].copy()

        catchment_geoms = gpd.GeoSeries(
            chunk["catchment_geometry"],
            crs=AREA_CRS,
            index=chunk.index
        )

        inter = chunk.geometry.intersection(catchment_geoms)
        areas.append(inter.area.to_numpy())

    return np.concatenate(areas)


# ------------------------------------------------------------
# Run by VPU to control memory
# ------------------------------------------------------------
parts = []
t0 = time.time()

vpus = sorted(selected_catchments["vpuid"].dropna().astype(str).unique())

print(f"VPUs to process: {len(vpus)}")
print(vpus)

for vpu in vpus:

    out_file = FRAC_SELECTED_CACHE / f"selected_fractional_crosswalk_vpu_{vpu}.parquet"

    if out_file.exists():
        df_cached = pd.read_parquet(out_file)
        parts.append(df_cached)
        print(f"{vpu}: cached ({len(df_cached):,} rows)")
        continue

    print(f"\nWorking on VPU {vpu}...")

    sub = selected_catchments[
        selected_catchments["vpuid"].astype(str) == vpu
    ].copy()

    if len(sub) == 0:
        print(f"{vpu}: no selected catchments")
        continue

    sub = sub.to_crs(AREA_CRS)

    try:
        sub["geometry"] = sub.geometry.make_valid()
    except Exception:
        sub["geometry"] = sub.geometry.buffer(0)

    divide_ids_vpu = sub["divide_id"].dropna().unique()

    # --------------------------------------------------------
    # Candidate cells from center-based crosswalk + neighbors
    # --------------------------------------------------------
    cw_vpu = selected_crosswalk[
        selected_crosswalk["divide_id"].isin(divide_ids_vpu)
    ][["divide_id", "cell_id", "row", "col"]].copy()

    candidate_cells = add_neighbor_cells(
        cw_vpu[["row", "col"]],
        pad=PAD_CELLS
    )

    # --------------------------------------------------------
    # Add cells for catchments with no center-based MRMS cell
    # --------------------------------------------------------
    catchments_with_center_cell = set(cw_vpu["divide_id"].dropna().unique())
    missing_center_catchments = sub[
        ~sub["divide_id"].isin(catchments_with_center_cell)
    ].copy()

    print(f"  selected catchments: {len(sub):,}")
    print(f"  catchments without center MRMS cell: {len(missing_center_catchments):,}")

    extra_cells = []

    if len(missing_center_catchments) > 0:
        missing_4326 = missing_center_catchments.to_crs(4326)

        for geom in missing_4326.geometry:
            west, south, east, north = geom.bounds
            extra_cells.append(
                rows_cols_for_bbox(west, south, east, north, pad=PAD_CELLS)
            )

    if len(extra_cells) > 0:
        extra_cells = pd.concat(extra_cells, ignore_index=True)
        candidate_cells = pd.concat(
            [candidate_cells, extra_cells],
            ignore_index=True
        )

    candidate_cells = (
        candidate_cells
        .drop_duplicates(subset=["row", "col"])
        .reset_index(drop=True)
    )

    print(f"  candidate MRMS cells: {len(candidate_cells):,}")

    if len(candidate_cells) == 0:
        print(f"{vpu}: no candidate cells")
        continue

    # --------------------------------------------------------
    # Build MRMS cell polygons
    # --------------------------------------------------------
    mrms_cells = build_mrms_cells_from_rows_cols(
        candidate_cells["row"].values,
        candidate_cells["col"].values
    )

    # --------------------------------------------------------
    # Spatial join: MRMS cell polygons intersect catchments
    # --------------------------------------------------------
    join_cols = [
        "divide_id",
        "vpuid",
        "nexus_id",
        "nexus_type",
        "is_terminal",
        "geometry"
    ]

    if "terminal_nexus_id" in sub.columns:
        join_cols.insert(-1, "terminal_nexus_id")

    join_cols = [c for c in join_cols if c in sub.columns]

    sub_join = sub[join_cols].copy()

    candidates = gpd.sjoin(
        mrms_cells,
        sub_join,
        how="inner",
        predicate="intersects"
    )

    if len(candidates) == 0:
        print(f"{vpu}: no intersecting cells")
        continue

    print(f"  candidate cell/catchment pairs: {len(candidates):,}")

    # Attach catchment geometry
    catchment_geom_lookup = sub_join.geometry.rename("catchment_geometry")

    candidates = candidates.join(
        catchment_geom_lookup,
        on="index_right"
    )

    # --------------------------------------------------------
    # True intersection area
    # --------------------------------------------------------
    candidates["intersection_area_m2"] = intersection_area_chunked(
        candidates,
        chunk_size=150_000
    )

    candidates = candidates[
        candidates["intersection_area_m2"] > 0.01
    ].copy()

    candidates["fraction_inside"] = (
        candidates["intersection_area_m2"] / candidates["cell_area_m2"]
    ).clip(0, 1)

    # --------------------------------------------------------
    # Save useful columns
    # --------------------------------------------------------
    keep_cols = [
        "cell_id",
        "row",
        "col",
        "lon",
        "lat",
        "divide_id",
        "vpuid",
        "nexus_id",
        "cell_area_m2",
        "intersection_area_m2",
        "fraction_inside"
    ]

    optional_cols = ["nexus_type", "is_terminal", "terminal_nexus_id"]

    for c in optional_cols:
        if c in candidates.columns and c not in keep_cols:
            keep_cols.append(c)

    df = (
        pd.DataFrame(candidates[keep_cols])
        .drop_duplicates(subset=["divide_id", "cell_id"])
        .sort_values(["divide_id", "cell_id"])
        .reset_index(drop=True)
    )

    # Normalize to weights that sum to 1 for each catchment
    # (kept for reference/QA only -- Notebook 4 re-normalizes over
    #  whichever cells actually have data after dropping NaNs per timestep,
    #  rather than using this pre-baked weight directly)
    df["weight"] = (
        df["fraction_inside"] /
        df.groupby("divide_id")["fraction_inside"].transform("sum")
    )

    df.to_parquet(out_file, index=False)
    parts.append(df)

    print(f"  final rows: {len(df):,}")
    print(f"  catchments hit: {df['divide_id'].nunique():,}")
    print(f"  saved: {out_file}")


# ------------------------------------------------------------
# Combine all selected VPU fractional files
# ------------------------------------------------------------
selected_fractional_crosswalk = (
    pd.concat(parts, ignore_index=True)
    .drop_duplicates(subset=["divide_id", "cell_id"])
    .reset_index(drop=True)
)

selected_fractional_crosswalk["weight"] = (
    selected_fractional_crosswalk["fraction_inside"] /
    selected_fractional_crosswalk.groupby("divide_id")["fraction_inside"].transform("sum")
)

selected_fractional_crosswalk.to_parquet(SELECTED_FRAC_FILE, index=False)

print("\n============================================================")
print("FRACTIONAL CROSSWALK READY \u2705")
print("============================================================")
print(f"rows               : {len(selected_fractional_crosswalk):,}")
print(f"unique catchments  : {selected_fractional_crosswalk['divide_id'].nunique():,}")
print(f"unique MRMS cells  : {selected_fractional_crosswalk['cell_id'].nunique():,}")
print(f"saved file         : {SELECTED_FRAC_FILE}")
print(f"total time         : {(time.time() - t0) / 60:.1f} minutes")

weight_check = selected_fractional_crosswalk.groupby("divide_id")["weight"].sum()
print("\nWeight sum check (should all be ~1.0):")
print(weight_check.describe())

VPUs to process: 1
['03N']
03N: cached (20,314 rows)

FRACTIONAL CROSSWALK READY ✅
rows               : 20,314
unique catchments  : 1,072
unique MRMS cells  : 12,576
saved file         : mrms_crosswalk_cache/step3_event_relevant/selected_fractional_crosswalk.parquet
total time         : 0.0 minutes

Weight sum check (should all be ~1.0):
count    1.072000e+03
mean     1.000000e+00
std      4.526114e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: weight, dtype: float64


## Save outputs for Notebook 4

Notebook 4 needs: the event manifest (window + recording gage per event), the
long-form event\u2192catchment table, the fractional crosswalk, and the shard
directory the rainfall actually lives in. `fraction_inside` (not the
pre-normalized `weight` column) is what Notebook 4 should use, re-normalizing
per timestep after dropping any cells that come back NaN \u2014 see the data
conventions note in the cell above.

In [13]:
OUT_DIR3 = Path("event_mrms_outputs")
OUT_DIR3.mkdir(exist_ok=True)

manifest_out.to_parquet(OUT_DIR3 / "manifest.parquet", index=False)
event_catchment_windows.to_parquet(OUT_DIR3 / "event_catchment_windows.parquet", index=False)
selected_fractional_crosswalk.to_parquet(OUT_DIR3 / "selected_fractional_crosswalk.parquet", index=False)

with open(OUT_DIR3 / "shard_dir.txt", "w") as f:
    f.write(str(SHARD_DIR.resolve()))

print("Saved to", OUT_DIR3.resolve())
for f in sorted(OUT_DIR3.iterdir()):
    print(" -", f.name, f"({f.stat().st_size / 1e6:.2f} MB)")

Saved to /home/jovyan/Group_Project/event_mrms_outputs
 - event_catchment_windows.parquet (0.01 MB)
 - manifest.parquet (0.01 MB)
 - selected_fractional_crosswalk.parquet (0.86 MB)
 - shard_dir.txt (0.00 MB)


## Notebook 3 \u2014 done \u2705

- Gage-basin event manifest built: recording gage + 5-day window + upstream
  contributing catchments per event.
- MRMS PrecipRate downloaded via `build_store()` into resumable day shards.
- Fractional MRMS\u2192catchment crosswalk built for exactly the gage-basin
  catchments needed.
- Everything saved to `event_mrms_outputs/` for **Notebook 4**.

**Next: Notebook 4** \u2014 for each event, read the relevant day shards, extract
the fractional-crosswalk cells, area-weight to per-catchment 2-min rainfall
rate (`\u03a3(rate \u00d7 fraction_inside) / \u03a3(fraction_inside)`, re-normalized over
non-NaN cells per timestep), resample to 15-min mean rate, convert to depth
(`rate15 \u00d7 0.25`), and export per-storm NetCDF forcing.